# 🫀 실험 10 — 유도 구성 ablation: **평균이 가린 것을 꺼낸다**

**MedKOS / `notebooks/exp10_lead_ablation.ipynb`** · 퀘스트 `ailab-2026-0015`

---

## 실험3이 남긴 빚

실험3에서 우리는 **MI·HYP를 뺐습니다. 7,300건, 전체의 33%.**
이유는 정당했습니다 — lead II로는 관측이 불가능하니 억지로 맞히라고 하면 환각을 배웁니다
(`ailab-2026-0016` 2-bis). 하지만 그건 **"지금은 못 본다"** 이지 **"영영 못 본다"** 가 아닙니다.

> **이 실험은 그 7,300건을 되찾아오는 실험입니다.**

그리고 되찾는 과정에서 훨씬 중요한 걸 봅니다.

## 주가설 — 유도의 이득은 라벨군마다 다르다 (교호작용)

CinC 2021 같은 축소유도 챌린지는 성능을 **전체 평균 하나로** 보고합니다.
그러면 이런 일이 벌어집니다:

> "2유도가 12유도의 95% 성능을 낸다" → **평균은 참인데, MI 환자에게는 거짓.**

리듬·전도장애는 유도를 늘려도 별로 안 오릅니다(이미 lead II로 보이니까).
경색·비대는 유도를 늘리면 크게 오릅니다(횡단면이 열리니까).
**둘을 평균 내면 "조금 오른다"가 되고, 임상적으로 가장 중요한 정보가 사라집니다.**

그래서 이 실험의 주지표는 성능이 아니라 **교호작용**입니다:

```
Δ_흉부군   = F1(MI·HYP,      {12}) − F1(MI·HYP,      {II})
Δ_전두면군 = F1(NORM·CD·STTC, {12}) − F1(NORM·CD·STTC, {II})

교호작용 = Δ_흉부군 − Δ_전두면군      ← 이게 0보다 유의하게 커야 한다
```

**사전등록 예측**

| # | 예측 | 근거 |
|---|---|---|
| **P1** | 교호작용 > 0 **유의** | 흉부유도가 여는 것은 횡단면이고, MI·HYP가 거기 산다 |
| **P2** | **`{I,II}` 는 MI·HYP를 거의 못 올린다** (`{II}`와 비슷) | `{I,II}`가 여는 건 **전두면 전체**뿐 — 횡단면은 한 줄도 안 열린다 |
| **P3** | `{II,V1}` 는 MI·HYP를 **부분적으로만** 올린다 (`{12}`에 못 미침) | V1은 전중격 하나. 전벽·측벽·후벽 국소화는 여전히 불가 |
| **P4** | lead-agnostic 모델의 `{12}` ≈ 유도고정 `{12}` 모델 | 실험9의 성공 기준. 이게 깨지면 유도 마스킹이 용량을 잡아먹은 것 |

**P2가 이 실험에서 가장 값진 예측입니다.** 왜냐면 `{I,II}`는 사지유도 6개와 정보량이
같은데(III = II − I …), 그럼에도 MI를 못 잡는다면 **"유도를 늘리면 좋아진다"가 아니라
"어느 평면을 여느냐가 전부다"** 가 증명되기 때문입니다. 유도 개수는 대리변수일 뿐입니다.

## 실험9를 같이 태운다

퀘스트 큐에서 실험10은 "실험9 가중치 하나로 유도만 교체"였습니다. 실험9(lead-agnostic
인코더)를 아직 안 돌렸으므로 **두 계열을 한 번에** 태웁니다:

| 계열 | 모델 수 | 답하는 질문 |
|---|---|---|
| **유도고정** | 4개 (구성마다 하나) | 각 구성의 **상한**은 얼마인가 (실험10) |
| **lead-agnostic (RLM)** | **1개** | 한 모델이 가변 유도를 받나 (실험9) |

Random Lead Masking = 배치마다 유도 구성을 무작위로 골라 나머지를 0으로 만듭니다.
그러면 한 모델이 `{II}`로도 `{12}`로도 추론할 수 있습니다 — 웨어러블과 텔레메트리를
**같은 가중치로** 서비스한다는 배치 시나리오가 이겁니다.

---

## 실험3에서 가져오는 교훈 3개

1. **동작점을 맞추고 비교한다.** 유도를 늘리면 결정경계가 같이 움직인다 — argmax 비교는
   성능 변화와 동작점 변화를 섞는다(실험2·3에서 두 번 걸렸다).
2. **오염을 먼저 점검한다.** 실험3에서 AFIB이 진단 라벨을 따라가는 걸 학습 **전에** 잡았다
   (NORM 0.3% · STTC 17.7%). MI·HYP가 새로 들어오니 같은 점검을 다시 한다.
3. **체크포인트를 남긴다.** (구성 × seed)마다 확률을 Drive에 저장한다. 런타임이 죽어도
   이어서 돌린다 — 실험1′에서 한 번 잃었다.


In [ ]:
# CELL 1 — Drive lib 재사용 + 설정
!pip -q install wfdb

import os, sys, json, time, ast, numpy as np
try:
    from google.colab import drive; drive.mount("/content/drive", force_remount=False)
    DRIVE_ROOT = "/content/drive/MyDrive"
except Exception as e:
    print("⚠️ Colab 아님:", e); DRIVE_ROOT = "/content"
PROJECT = os.path.join(DRIVE_ROOT, "MedKOS", "ecg-model")
sys.path.insert(0, os.path.join(PROJECT, "lib"))
from medkos_run import MedKOSRun

QUICK   = True        # True: 클래스당 1200건 / False: 전체
FS      = 100
# ★ 실험3이 뺐던 MI·HYP를 되찾아온다 — 이 실험의 존재 이유
CLASSES  = ["NORM", "CD", "STTC", "MI", "HYP"]
G_FRONT  = ["NORM", "CD", "STTC"]   # 전두면군 — lead II로 관측 가능(실험3에서 확인)
G_PRECOR = ["MI", "HYP"]            # 흉부군 — lead II로 관측 불가
LEADS12  = ["I", "II", "III", "AVR", "AVL", "AVF",
            "V1", "V2", "V3", "V4", "V5", "V6"]
CONFIGS = {                          # 이름 → LEADS12 인덱스
    "II":    [1],                    # 웨어러블 단일유도
    "I+II":  [0, 1],                 # 웨어러블 2유도 ≡ 사지유도 6개 전체
    "II+V1": [1, 6],                 # 임상 텔레메트리 표준
    "12":    list(range(12)),        # 임상 12유도
}
CAP = 1200 if QUICK else 10**9
N_SEEDS, EPOCHS = (2, 12) if QUICK else (3, 25)
SEED0 = 20260801

CONFIG = dict(exp="exp10_lead_ablation", quest="ailab-2026-0015",
              hypothesis="유도 추가의 이득은 라벨군마다 다르다(교호작용 > 0)",
              prediction=("P1 교호작용>0 유의 · P2 {I,II}는 MI·HYP 거의 못 올림 · "
                          "P3 {II,V1}는 부분만 · P4 agnostic{12} ≈ 고정{12}"),
              primary_metric="interaction = Δ흉부군 − Δ전두면군",
              dataset="ptb-xl 1.0.3 records100", classes=CLASSES,
              group_frontal=G_FRONT, group_precordial=G_PRECOR,
              lead_configs={k: [LEADS12[i] for i in v] for k, v in CONFIGS.items()},
              split="official strat_fold 1-8/9/10", fs=FS, cap_per_class=CAP,
              n_seeds=N_SEEDS, epochs=EPOCHS, seed0=SEED0, quick=QUICK, boot=2000)
np.random.seed(SEED0)
try:
    import matplotlib, matplotlib.font_manager as fm, subprocess
    if not any("Nanum" in f.name for f in fm.fontManager.ttflist):
        subprocess.run(["apt-get", "-qq", "install", "-y", "fonts-nanum"], check=False)
        fm._load_fontmanager(try_read_cache=False)
    matplotlib.rc("font", family="NanumGothic"); matplotlib.rc("axes", unicode_minus=False)
except Exception as e:
    print("한글 폰트 설정 건너뜀:", e)

run = MedKOSRun(PROJECT, "exp10_lead_ablation", CONFIG)
run.log(f"유도 구성 {len(CONFIGS)}종 · 클래스 {CLASSES}")
run.log(f"모델 {len(CONFIGS)}개(유도고정) + 1개(lead-agnostic) × seed {N_SEEDS}"
        f" = 총 {(len(CONFIGS)+1)*N_SEEDS}회 학습 — QUICK에서 60~90분 예상")


### CELL 2 — PTB-XL 확보 (실험3과 같은 데이터·같은 캐시)

실험3을 돌린 세션이면 `/content/ptbxl`이 이미 있을 수 있지만, 런타임이 바뀌었으면
다시 받습니다(로컬 디스크에 풀고 Drive에는 최종 npz 하나만 남깁니다).


In [ ]:
# CELL 2 — PTB-XL 확보 + 구조 확인
import wfdb, pandas as pd, subprocess, zipfile, shutil

PTB = "/content/ptbxl"
os.makedirs(PTB, exist_ok=True)
BASE = "https://physionet.org/files/ptb-xl/1.0.3"
ZIP = ("https://physionet.org/static/published-projects/ptb-xl/"
       "ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3.zip")

def fetch(url, dest):
    if os.path.exists(dest) and os.path.getsize(dest) > 0:
        return True
    r = subprocess.run(["wget", "-q", "-O", dest, url])
    return r.returncode == 0 and os.path.getsize(dest) > 0

for f in ("ptbxl_database.csv", "scp_statements.csv"):
    ok = fetch(f"{BASE}/{f}", os.path.join(PTB, f))
    run.log(f"{'✅' if ok else '❌'} {f}")

df = pd.read_csv(os.path.join(PTB, "ptbxl_database.csv"), index_col="ecg_id")
scp = pd.read_csv(os.path.join(PTB, "scp_statements.csv"), index_col=0)
run.log(f"레코드 {len(df):,} · 환자 {df.patient_id.nunique():,}")

CACHE = run.data(f"ptbxl_12lead_{'quick' if QUICK else 'full'}.npz")
if os.path.exists(CACHE):
    run.log("최종 npz 캐시가 있어 신호 다운로드를 건너뜁니다")
elif not os.path.isdir(os.path.join(PTB, "records100")):
    z = "/content/ptbxl.zip"
    run.log("전체 zip 내려받는 중(약 1.7GB, 5~15분)…")
    if not fetch(ZIP, z):
        raise RuntimeError("zip 다운로드 실패 — 네트워크 확인")
    run.log("압축 푸는 중…")
    with zipfile.ZipFile(z) as zf:
        zf.extractall("/content/_ptb")
    root = next(p for p, d, _ in os.walk("/content/_ptb") if "records100" in d)
    for name in os.listdir(root):
        src, dst = os.path.join(root, name), os.path.join(PTB, name)
        if not os.path.exists(dst):
            shutil.move(src, dst)
    os.remove(z); shutil.rmtree("/content/_ptb", ignore_errors=True)
    run.log("✅ records100 준비 완료")
else:
    run.log("records100 이미 있음")

LEAD_ORDER = None
if os.path.isdir(os.path.join(PTB, "records100")):
    s = wfdb.rdrecord(os.path.join(PTB, df.iloc[0].filename_lr))
    got = [n.strip().upper() for n in s.sig_name]
    run.log(f"샘플 유도 {got}")
    missing = [n for n in LEADS12 if n not in got]
    if missing:
        raise RuntimeError(f"없는 유도: {missing}")
    # 순서가 다를 수 있으므로 assert로 죽이지 말고 이름으로 재정렬한다.
    # 이후 CONFIGS의 인덱스는 항상 LEADS12 순서를 가리킨다.
    LEAD_ORDER = [got.index(n) for n in LEADS12]
    run.log(f"{'✅ 순서 일치' if LEAD_ORDER == list(range(12)) else '↩️ 재정렬 적용'}"
            f" — CONFIGS 인덱스는 LEADS12 기준")


In [ ]:
# CELL 3 — 라벨: 5개 superclass 전부 (★ MI·HYP 복귀)
agg = scp[scp.diagnostic == 1].diagnostic_class.to_dict()

def superclasses(codes_str):
    d = ast.literal_eval(codes_str)
    return sorted({agg[k] for k in d if k in agg})

df["sc"] = df.scp_codes.apply(superclasses)
df["n_sc"] = df.sc.apply(len)

# 실험3과 같은 규칙: 단일 superclass만. 단 MI·HYP를 빼지 않는다.
# (pandas의 &는 단축평가를 안 하므로 len 검사를 람다 안에 둔다 — 실험3 CELL 3 버그)
keep = df.sc.apply(lambda s: len(s) == 1 and s[0] in CLASSES)
sub = df[keep].copy()
sub["y"] = sub.sc.apply(lambda s: CLASSES.index(s[0]))

run.log(f"전체 {len(df):,}건")
run.log(f"  진단 superclass 없음 {int((df.n_sc==0).sum()):,}건 · "
        f"다중 superclass {int((df.n_sc>1).sum()):,}건")
run.log(f"→ 단일 superclass 5종 → **{len(sub):,}건** "
        f"(실험3의 13,177건 + MI·HYP {len(sub)-13177:,}건)")
for i, c in enumerate(CLASSES):
    m = sub.y == i
    tag = "흉부군" if c in G_PRECOR else "전두면군"
    run.log(f"  {c:5s}({tag}): {int(m.sum()):6,}건 · 환자 {sub[m].patient_id.nunique():,}명")

if QUICK:
    sub = pd.concat([g.sample(min(len(g), CAP), random_state=SEED0)
                     for _, g in sub.groupby("y")])
    run.log(f"QUICK: 클래스별 최대 {CAP}건 → {len(sub):,}건")
run.log(f"fold 분포: {sub.strat_fold.value_counts().sort_index().to_dict()}")


In [ ]:
# CELL 4 — 12유도 신호 로드 (float16 저장 — 12배 커지므로)
CACHE = run.data(f"ptbxl_12lead_{'quick' if QUICK else 'full'}.npz")
if os.path.exists(CACHE):
    z = np.load(CACHE, allow_pickle=True)
    X, Y, FOLD, PID, EID = z["X"], z["y"], z["fold"], z["pid"], z["eid"]
    run.log(f"캐시 재사용: {X.shape} ({X.nbytes/1e6:.0f} MB, {X.dtype})")
    # 캐시가 지금 sub와 같은 표본인지 확인 — 실험3에서 낡은 캐시에 한 번 데였다
    if len(X) != len(sub) or not np.array_equal(np.sort(EID), np.sort(sub.index.values)):
        raise RuntimeError("캐시가 지금 sub와 다르다. CACHE 파일을 지우고 다시 실행할 것")
    sub = sub.loc[EID]                       # 캐시 순서에 sub를 맞춘다
else:
    Xs, Ys, Fs_, Ps, Es, nofail = [], [], [], [], [], 0
    t0 = time.time()
    for k, (eid, row) in enumerate(sub.iterrows()):
        try:
            rec = wfdb.rdrecord(os.path.join(PTB, row.filename_lr))
        except Exception:
            nofail += 1; continue
        names = [n.strip().upper() for n in rec.sig_name]
        if any(n not in names for n in LEADS12):
            nofail += 1; continue
        order = [names.index(n) for n in LEADS12]             # 항상 LEADS12 순서로
        s = np.nan_to_num(rec.p_signal)[:, order].astype("float64")   # (1000, 12)
        # ★ 유도별이 아니라 레코드 전체 스케일로 정규화한다.
        #   유도별로 나누면 유도 간 '상대 진폭'이 지워지는데, 저전압·R파증가처럼
        #   진폭 비교가 본질인 소견이 통째로 사라진다.
        q = np.percentile(s, 75) - np.percentile(s, 25)
        s = (s - np.median(s)) / (q + 1e-6)
        Xs.append(np.clip(s, -20, 20).astype("float16"))
        Ys.append(int(row.y)); Fs_.append(int(row.strat_fold))
        Ps.append(int(row.patient_id)); Es.append(int(eid))
        if (k + 1) % 1000 == 0:
            run.log(f"  {k+1}/{len(sub)} ({time.time()-t0:.0f}s)")
    X = np.array(Xs, "float16"); Y = np.array(Ys)
    FOLD = np.array(Fs_); PID = np.array(Ps); EID = np.array(Es)
    np.savez_compressed(CACHE, X=X, y=Y, fold=FOLD, pid=PID, eid=EID)
    run.log(f"저장: {CACHE} ({X.nbytes/1e6:.0f} MB, 실패 {nofail}건)")
    sub = sub.loc[EID]

run.log(f"\nX{X.shape} · 클래스 {np.bincount(Y, minlength=5).tolist()} ({CLASSES})")
assert len(sub) == len(Y) and (sub.y.values == Y).all(), "sub와 배열 정렬 불일치"


### CELL 5 — 물리 사전점검: `{I,II}` 가 정말 사지유도 6개와 같은가

이 실험의 P2는 **"`{I,II}`는 사지유도 전체와 정보량이 같다"** 는 심전도 기하학에 기댑니다:

```
III = II − I     aVR = −(I+II)/2     aVL = I − II/2     aVF = II − I/2
```

**말로만 믿지 말고 데이터에서 확인합니다.** 이게 성립하면 `{I,II}`에 III·aVR·aVL·aVF를
더하는 건 **증명 가능하게 무의미**하고, P2("`{I,II}`도 MI를 못 잡는다")는 "유도를 덜 줘서"가
아니라 **"전두면에는 그 정보가 없어서"** 라는 해석이 확정됩니다.


In [ ]:
# CELL 5 — 사지유도 선형종속 검증 (전두면 기하학)
IDX = {n: i for i, n in enumerate(LEADS12)}
def L(a, name):
    """(n, 1000, 12) 배열에서 유도 하나를 (n, 1000)으로 꺼낸다."""
    return a[:, :, IDX[name]]
chk = {
    "III  = II − I":       ("III", lambda a: L(a, "II") - L(a, "I")),
    "aVR  = −(I+II)/2":    ("AVR", lambda a: -(L(a, "I") + L(a, "II")) / 2),
    "aVL  = I − II/2":     ("AVL", lambda a: L(a, "I") - L(a, "II") / 2),
    "aVF  = II − I/2":     ("AVF", lambda a: L(a, "II") - L(a, "I") / 2),
}
smp = np.random.RandomState(SEED0).choice(len(X), min(400, len(X)), replace=False)
A = X[smp].astype("float32")                     # (n, 1000, 12)
scale = float(np.percentile(np.abs(A), 99))      # 신호의 대표 진폭
run.log(f"표본 {len(smp)}건 · 신호 99퍼센타일 진폭 = {scale:.3f} (정규화 단위)")
geo = {}
for name, (tgt, fn) in chk.items():
    err = np.abs(L(A, tgt) - fn(A))
    rel = float(err.mean() / (scale + 1e-9))
    geo[name] = {"mean_abs_err": float(err.mean()), "rel": rel}
    ok = "✅" if rel < 0.02 else ("⚠️" if rel < 0.10 else "❌")
    run.log(f"  {ok} {name:18s} 평균오차 {err.mean():.4f} = 진폭의 {rel*100:5.2f}%")

worst = max(v["rel"] for v in geo.values())
run.log("")
if worst < 0.02:
    run.log("→ **선형종속 확인.** {I,II}에 III·aVR·aVL·aVF를 더하는 것은 증명 가능하게")
    run.log("   무의미하다. P2가 참이면 그건 '유도를 덜 줘서'가 아니라")
    run.log("   **'전두면에 그 정보가 없어서'** 다.")
else:
    run.log(f"→ ⚠️ 최대 상대오차 {worst*100:.1f}% — 이 데이터셋은 사지유도를 독립 측정했거나")
    run.log("   전처리가 개입했다. P2의 해석에 이 점을 함께 적을 것.")
run.save_json("precheck_lead_geometry", {"rel_errors": geo, "worst_rel": worst})


In [ ]:
# CELL 5.5 — 오염 점검 (실험3의 교훈: 리듬 코드는 진단 라벨과 독립 축이다)
from collections import Counter
rhy_codes = set(scp[scp.rhythm == 1].index)
rl = [sorted(k for k in ast.literal_eval(s) if k in rhy_codes) for s in sub.scp_codes]
is_sr = np.array([ks == ["SR"] for ks in rl])

run.log("클래스별 리듬 코드 (MI·HYP가 새로 들어왔으니 다시 본다)")
cont = {}
for i, c in enumerate(CLASSES):
    m = Y == i
    cnt = Counter(k for j in np.where(m)[0] for k in rl[j])
    afib = cnt.get("AFIB", 0)
    cont[c] = {"pure_sr": float(is_sr[m].mean()), "afib_rate": afib / max(int(m.sum()), 1)}
    run.log(f"  {c:5s} n={int(m.sum()):5,} · 순수SR {is_sr[m].mean()*100:5.1f}%"
            f" · AFIB {afib:4,}건({afib/max(int(m.sum()),1)*100:4.1f}%)"
            f" · {', '.join(f'{k} {v}' for k, v in cnt.most_common(3))}")

spread = max(v["afib_rate"] for v in cont.values()) - min(v["afib_rate"] for v in cont.values())
run.log(f"\nAFIB 비율 최대-최소 격차 = {spread*100:.1f}%p")
run.log("  이 실험의 타깃은 리듬이 아니라 유도이므로 오염이 주가설을 직접 위협하진 않는다.")
run.log("  다만 클래스별 난이도에 영향을 주므로 기록해 둔다(실험3: STTC 17.7% vs NORM 0.3%).")
run.save_json("precheck_contamination", {"per_class": cont, "afib_spread": spread})


In [ ]:
# CELL 6 — 학습: 유도고정 4개 + lead-agnostic 1개 (체크포인트 있음)
import tensorflow as tf
from tensorflow.keras import layers, models

tr = np.isin(FOLD, range(1, 9)); va = FOLD == 9; te = FOLD == 10
NC = len(CLASSES)
run.log(f"학습 {tr.sum():,} / 검증 {va.sum():,} / 테스트 {te.sum():,} (공식 fold)")
run.log(f"테스트 클래스 {np.bincount(Y[te], minlength=NC).tolist()}")

def auto_weights(y, beta=0.9999):
    w = {c: (1 - beta) / (1 - beta ** max((y == c).sum(), 1)) for c in range(NC)}
    return {c: float(v / w[0]) for c, v in w.items()}
CW = auto_weights(Y[tr]); SW = np.array([CW[int(c)] for c in Y[tr]], "float32")
run.log(f"가중치 { {CLASSES[c]: round(v,2) for c,v in CW.items()} }")

def backbone(inp):
    x = inp
    for f, k in ((32, 9), (64, 7), (128, 5), (128, 3)):
        x = layers.Conv1D(f, k, padding="same", activation="relu")(x)
        x = layers.BatchNormalization()(x); x = layers.MaxPooling1D(2)(x)
    return layers.GlobalAveragePooling1D()(x)

def build(n_ch, seed, agnostic=False):
    tf.keras.utils.set_random_seed(seed)
    si = layers.Input((X.shape[1], n_ch)); ins = [si]
    h = layers.Dense(64, activation="relu")(backbone(si))
    if agnostic:
        # 어떤 유도가 살아 있는지 모델에게 알려준다. 0으로 채운 채널과
        # '정말 0에 가까운 신호'를 구분하지 못하면 마스킹이 노이즈가 된다.
        mi = layers.Input((12,)); ins.append(mi)
        h = layers.Concatenate()([h, layers.Dense(16, activation="relu")(mi)])
    h = layers.Dropout(0.3)(h); h = layers.Dense(64, activation="relu")(h)
    m = models.Model(ins, layers.Dense(NC, activation="softmax")(h))
    m.compile(optimizer=tf.keras.optimizers.Adam(1e-3, clipnorm=1.0),
              loss="sparse_categorical_crossentropy")
    return m

def mask_of(cfg):
    m = np.zeros(12, "float32"); m[CONFIGS[cfg]] = 1.0
    return m

class RLMSeq(tf.keras.utils.Sequence):
    """배치마다 유도 구성을 무작위로 골라 나머지를 0으로 만든다(Random Lead Masking)."""
    def __init__(self, idx, bs, seed, shuffle=True):
        self.idx, self.bs, self.shuffle = idx, bs, shuffle
        self.names = list(CONFIGS); self.rs = np.random.RandomState(seed)
        self.order = idx.copy(); self.on_epoch_end()
    def __len__(self):
        return int(np.ceil(len(self.idx) / self.bs))
    def on_epoch_end(self):
        if self.shuffle: self.rs.shuffle(self.order)
    def __getitem__(self, i):
        b = self.order[i * self.bs:(i + 1) * self.bs]
        # 검증은 구성을 순환시켜 결정론적으로(=매 epoch 같은 잣대)
        cfg = self.names[self.rs.randint(len(self.names))] if self.shuffle \
              else self.names[i % len(self.names)]
        m = mask_of(cfg)
        xb = X[b].astype("float32") * m
        mb = np.repeat(m[None], len(b), 0)
        return (xb, mb), Y[b], SWALL[b]

SWALL = np.zeros(len(Y), "float32"); SWALL[tr] = SW
TR, VA, TE = np.where(tr)[0], np.where(va)[0], np.where(te)[0]

probs, t0 = {}, time.time()

# ── 계열 1: 유도고정 (구성마다 독립 모델)
for cfg, leads in CONFIGS.items():
    arm = f"fixed_{cfg}"
    c = run.load_arm(arm)
    if c is not None:
        probs[arm] = c; run.log(f"⏭ {arm} 이미 완료(체크포인트)"); continue
    acc = np.zeros((len(TE), NC))
    for s in range(N_SEEDS):
        m = build(len(leads), SEED0 + s)
        h = m.fit(X[TR][:, :, leads].astype("float32"), Y[TR],
                  validation_data=(X[VA][:, :, leads].astype("float32"), Y[VA]),
                  epochs=EPOCHS, batch_size=128, sample_weight=SW, verbose=0)
        pr = m.predict(X[TE][:, :, leads].astype("float32"), batch_size=512, verbose=0)
        acc += pr
        run.log(f"  {arm} seed {s+1}/{N_SEEDS} ({time.time()-t0:.0f}s) "
                f"loss {h.history['loss'][0]:.3f}→{h.history['loss'][-1]:.3f} "
                f"val {h.history['val_loss'][-1]:.3f} | "
                f"예측분포 {np.bincount(pr.argmax(1), minlength=NC).tolist()}")
        tf.keras.backend.clear_session()
    probs[arm] = acc / N_SEEDS
    run.save_arm(arm, probs[arm])

# ── 계열 2: lead-agnostic 1개 → 구성마다 평가 (실험9)
AG = {}
need = [f"agno_{c}" for c in CONFIGS if run.load_arm(f"agno_{c}") is None]
if not need:
    for cfg in CONFIGS:
        AG[cfg] = run.load_arm(f"agno_{cfg}")
    run.log("⏭ lead-agnostic 전부 완료(체크포인트)")
else:
    acc = {cfg: np.zeros((len(TE), NC)) for cfg in CONFIGS}
    for s in range(N_SEEDS):
        m = build(12, SEED0 + 100 + s, agnostic=True)
        h = m.fit(RLMSeq(TR, 128, SEED0 + s),
                  validation_data=RLMSeq(VA, 256, SEED0, shuffle=False),
                  epochs=EPOCHS, verbose=0)
        run.log(f"  agnostic seed {s+1}/{N_SEEDS} ({time.time()-t0:.0f}s) "
                f"loss {h.history['loss'][0]:.3f}→{h.history['loss'][-1]:.3f} "
                f"val {h.history['val_loss'][-1]:.3f}")
        for cfg in CONFIGS:
            mk = mask_of(cfg)
            pr = m.predict([X[TE].astype("float32") * mk,
                            np.repeat(mk[None], len(TE), 0)], batch_size=512, verbose=0)
            acc[cfg] += pr
            run.log(f"    └ {cfg:6s} 예측분포 {np.bincount(pr.argmax(1), minlength=NC).tolist()}")
        tf.keras.backend.clear_session()
    for cfg in CONFIGS:
        AG[cfg] = acc[cfg] / N_SEEDS
        run.save_arm(f"agno_{cfg}", AG[cfg])
for cfg in CONFIGS:
    probs[f"agno_{cfg}"] = AG[cfg] if cfg in AG else run.load_arm(f"agno_{cfg}")
run.log(f"\n총 {time.time()-t0:.0f}s · arm {len(probs)}개")


In [ ]:
# CELL 7 — 평가: 라벨군별 표 + 교호작용 (주지표)
from sklearn.metrics import f1_score, confusion_matrix
yte = Y[TE]
GF = [CLASSES.index(c) for c in G_FRONT]
GP = [CLASSES.index(c) for c in G_PRECOR]

def per_class_f1(pred):
    return f1_score(yte, pred, average=None, labels=range(NC), zero_division=0)

def gf1(pred, grp):
    """라벨군 macro-F1 = 그 군에 속한 클래스 F1의 평균."""
    f = per_class_f1(pred)
    return float(np.mean([f[i] for i in grp]))

res = {}
for arm, pr in probs.items():
    p = pr.argmax(1)
    res[arm] = {"pred": p, "f1": per_class_f1(p),
                "macro": float(f1_score(yte, p, average="macro", zero_division=0)),
                "front": gf1(p, GF), "precor": gf1(p, GP)}

# ── 표 1: CinC 스타일 — 전체 평균 하나 (이게 무엇을 가리는지 보여주려고 먼저 찍는다)
run.log("=" * 88)
run.log("【표 1】 전체 평균만 보고할 때 — 축소유도 챌린지들이 쓰는 방식")
run.log("=" * 88)
run.log(f"{'유도구성':<10}{'유도고정 macro-F1':>20}{'12유도 대비':>14}")
base12 = res["fixed_12"]["macro"]
for cfg in CONFIGS:
    r = res[f"fixed_{cfg}"]
    run.log(f"{cfg:<10}{r['macro']:>20.4f}{r['macro']/base12*100:>13.1f}%")
run.log("  ↑ 이 표만 보면 '적은 유도로도 대부분의 성능이 나온다'로 읽힌다.")

# ── 표 2: 라벨군별 (이 실험의 본론)
run.log("\n" + "=" * 88)
run.log("【표 2】 라벨군별로 쪼개면 — 평균이 가린 것")
run.log("=" * 88)
run.log(f"{'유도구성':<10}{'전두면군':>12}{'흉부군':>12}{'격차':>10}   " +
        "".join(f"{c:>8}" for c in CLASSES))
for cfg in CONFIGS:
    r = res[f"fixed_{cfg}"]
    run.log(f"{cfg:<10}{r['front']:>12.4f}{r['precor']:>12.4f}"
            f"{r['front']-r['precor']:>+10.4f}   " +
            "".join(f"{r['f1'][i]:>8.3f}" for i in range(NC)))
run.log(f"  전두면군 = {G_FRONT} · 흉부군 = {G_PRECOR}")

# ── 주지표: 교호작용
def boot_interaction(pred_hi, pred_lo, B=CONFIG["boot"], seed=SEED0):
    """(Δ흉부군 − Δ전두면군)의 부트스트랩 분포. 레코드 단위 재표본."""
    rs = np.random.RandomState(seed); n = len(yte); out = []
    for _ in range(B):
        i = rs.randint(0, n, n)
        y = yte[i]
        def g(p, grp):
            f = f1_score(y, p[i], average=None, labels=range(NC), zero_division=0)
            return np.mean([f[j] for j in grp])
        out.append((g(pred_hi, GP) - g(pred_lo, GP)) -
                   (g(pred_hi, GF) - g(pred_lo, GF)))
    out = np.array(out)
    return float(out.mean()), float(np.percentile(out, 2.5)), float(np.percentile(out, 97.5))

run.log("\n" + "=" * 88)
run.log("【주지표】 교호작용 = Δ흉부군 − Δ전두면군   (0보다 유의하게 커야 P1 적중)")
run.log("=" * 88)
inter = {}
for cfg in ("I+II", "II+V1", "12"):
    hi, lo = res[f"fixed_{cfg}"], res["fixed_II"]
    d_p, d_f = hi["precor"] - lo["precor"], hi["front"] - lo["front"]
    m_, l_, u_ = boot_interaction(hi["pred"], lo["pred"])
    sig = bool(l_ > 0 or u_ < 0)
    inter[cfg] = {"delta_precordial": d_p, "delta_frontal": d_f,
                  "interaction": m_, "ci": [l_, u_], "significant": sig}
    run.log(f"  {cfg:>6s} vs II │ Δ흉부 {d_p:+.4f} · Δ전두면 {d_f:+.4f} │ "
            f"교호작용 {m_:+.4f} [{l_:+.4f}, {u_:+.4f}] {'유의 ★' if sig else '비유의'}")

# ── 사전등록 예측 채점
run.log("\n" + "=" * 88)
run.log("【사전등록 예측 채점】")
run.log("=" * 88)
verd = {}
i12 = inter["12"]
verd["P1"] = bool(i12["significant"] and i12["interaction"] > 0)
run.log(f"  P1 교호작용({{12}} vs {{II}}) > 0 유의 → "
        f"{'✅ 적중' if verd['P1'] else '❌ 빗나감'} ({i12['interaction']:+.4f})")

d_p_i2 = inter["I+II"]["delta_precordial"]
verd["P2"] = bool(abs(d_p_i2) < 0.5 * max(i12["delta_precordial"], 1e-9))
run.log(f"  P2 {{I,II}}는 흉부군을 거의 못 올림 → "
        f"{'✅ 적중' if verd['P2'] else '❌ 빗나감'} "
        f"(Δ흉부 {d_p_i2:+.4f} vs {{12}}의 {i12['delta_precordial']:+.4f})")

d_p_v1 = inter["II+V1"]["delta_precordial"]
verd["P3"] = bool(0 < d_p_v1 < i12["delta_precordial"])
run.log(f"  P3 {{II,V1}}는 부분만 회복 → "
        f"{'✅ 적중' if verd['P3'] else '❌ 빗나감'} (Δ흉부 {d_p_v1:+.4f})")

ag12, fx12 = res["agno_12"]["macro"], res["fixed_12"]["macro"]
verd["P4"] = bool(abs(ag12 - fx12) < 0.05)
run.log(f"  P4 agnostic{{12}} ≈ 고정{{12}} → {'✅ 적중' if verd['P4'] else '❌ 빗나감'} "
        f"({ag12:.4f} vs {fx12:.4f}, Δ{ag12-fx12:+.4f})")

# ── 실험9: 한 모델이 가변 유도를 받나
run.log("\n" + "=" * 88)
run.log("【실험9】 lead-agnostic 1개 모델 vs 유도고정 4개 모델")
run.log("=" * 88)
run.log(f"{'유도구성':<10}{'고정':>10}{'agnostic':>12}{'차이':>10}"
        f"{'  고정 흉부':>12}{'agno 흉부':>12}")
ag_cmp = {}
for cfg in CONFIGS:
    f_, a_ = res[f"fixed_{cfg}"], res[f"agno_{cfg}"]
    ag_cmp[cfg] = {"fixed": f_["macro"], "agnostic": a_["macro"],
                   "diff": a_["macro"] - f_["macro"],
                   "fixed_precor": f_["precor"], "agno_precor": a_["precor"]}
    run.log(f"{cfg:<10}{f_['macro']:>10.4f}{a_['macro']:>12.4f}"
            f"{a_['macro']-f_['macro']:>+10.4f}"
            f"{f_['precor']:>12.4f}{a_['precor']:>12.4f}")
run.log("  차이가 −0.05보다 나쁘면 유도 마스킹이 모델 용량을 잡아먹은 것 →")
run.log("  '한 가중치로 웨어러블·텔레메트리 둘 다' 전략의 비용이 그만큼이다.")

# ── 동작점 정합 (실험2·3의 교훈)
run.log("\n[사후·사전등록 아님] 동작점을 {12} 기준 오경보율로 맞춘 뒤 재비교")
norm_m = (yte == 0)
def at_same_fa(prob, target):
    lo, hi = 0.02, 50.0
    for _ in range(40):
        mid = (lo * hi) ** 0.5
        p = prob.copy(); p[:, 0] *= mid
        if float((p.argmax(1)[norm_m] != 0).mean()) > target: lo = mid
        else: hi = mid
    p = prob.copy(); p[:, 0] *= hi
    return p.argmax(1), hi
# 기준은 {II} — 이 실험의 모든 Δ가 {II} 대비이므로 "단일유도와 같은 경보 부담에서
# 유도를 늘리면 얼마나 얻나"가 배치 질문에 정확히 대응한다.
ref = float((res["fixed_II"]["pred"][norm_m] != 0).mean())
matched = {}
if not (0.02 < ref < 0.95):
    run.log(f"  ⛔ 기준 오경보율이 극단({ref:.3f}) — 동작점 정합이 무의미하므로 생략")
    matched = {"status": "skipped", "ref_false_alarm": ref}
else:
    run.log(f"  기준 오경보율 = {{II}}의 {ref:.3f}")
    for cfg in CONFIGS:
        pk, al = at_same_fa(probs[f"fixed_{cfg}"], ref)
        matched[cfg] = {"macro": float(f1_score(yte, pk, average="macro", zero_division=0)),
                        "precor": gf1(pk, GP), "alpha": al}
        run.log(f"  {cfg:<7} macro {res[f'fixed_{cfg}']['macro']:.4f}(argmax) → "
                f"{matched[cfg]['macro']:.4f}(정합) · 흉부군 {matched[cfg]['precor']:.4f}"
                f" · α={al:.2f}")

# ── 최종 판정
hits = sum(verd.values())
if verd["P1"] and verd["P2"]:
    verdict = ("확증 — 유도의 이득은 라벨군마다 다르다. 전두면을 아무리 채워도 "
               "흉부군은 안 열린다. **유도 '개수'가 아니라 '평면'이 결정한다**")
elif verd["P1"]:
    verdict = ("부분 확증 — 교호작용은 유의하나 {I,II}도 흉부군을 올렸다. "
               "전두면/횡단면 이분법이 생각보다 덜 깨끗하다")
elif i12["interaction"] < 0:
    verdict = "예측 반전 — 유도 추가가 전두면군을 더 올렸다. 라벨·전처리 재점검 필요"
else:
    verdict = "미결 — 교호작용을 검출하지 못했다(검정력 또는 QUICK 표본)"
run.log("\n" + "=" * 88)
run.log(f"▶ 사전등록 예측 {hits}/4 적중")
run.log(f"▶ {verdict}")
run.log("=" * 88)

run.save_json("evaluation", {
    "per_arm": {k: {"macro": v["macro"], "frontal": v["front"], "precordial": v["precor"],
                    "f1_per_class": {CLASSES[i]: float(v["f1"][i]) for i in range(NC)},
                    "confusion": confusion_matrix(yte, v["pred"],
                                                  labels=range(NC)).tolist()}
                for k, v in res.items()},
    "interaction": inter, "predictions": verd, "agnostic_vs_fixed": ag_cmp,
    "matched_operating_point": matched, "verdict": verdict})

result = {"week": 2, "exp_id": "exp10_lead_ablation", "quest": "ailab-2026-0015",
          "task": "PTB-XL 유도 구성 ablation — 라벨군별 교호작용 (+실험9 lead-agnostic)",
          "split": "inter", "metric": "interaction_precordial_minus_frontal",
          "value": round(i12["interaction"], 4),
          "passed": bool(verd["P1"] and verd["P2"]),
          "date": time.strftime("%Y-%m-%d"),
          "arms": {k: round(v["macro"], 4) for k, v in res.items()},
          "interaction": inter, "predictions": verd, "verdict": verdict,
          "summary": (f"교호작용({{12}}vs{{II}}) {i12['interaction']:+.4f} "
                      f"[{i12['ci'][0]:+.4f},{i12['ci'][1]:+.4f}] · "
                      f"Δ흉부 {i12['delta_precordial']:+.4f} vs Δ전두면 "
                      f"{i12['delta_frontal']:+.4f} · 예측 {hits}/4 → "
                      f"{verdict.split(' —')[0]}")}
result = run.finish(result)
import shutil; shutil.copy(os.path.join(run.dir, "result.json"), "/content/result.json")
print(f"""
────────────────────────────────────────────────────────────────
📁 {run.dir}
  python pipelines/ingest_run.py --results result.json \\
      --notebook notebooks/exp10_lead_ablation.ipynb \\
      --quest ailab-2026-0015 --step "exp10-lead-ablation" \\
      --note "{result['summary']}"
────────────────────────────────────────────────────────────────""")


---

## 결과 읽는 법

| 교호작용 | 뜻 | 다음 |
|---|---|---|
| **> 0 유의 + P2 적중** | **예측 적중.** 유도 개수가 아니라 **평면**이 결정한다. 축소유도 논문의 "95% 성능" 주장이 라벨군을 평균 내서 생긴 착시임을 실증 | 배치 전략 확정: 웨어러블은 전두면군만 담당하고 흉부군은 **abstain**. `ailab-2026-0016`의 3-tier를 모델 출력에 직결 |
| > 0 유의하나 P2 빗나감 | `{I,II}`도 흉부군을 올렸다 | 전두면/횡단면 이분법이 덜 깨끗하다. 어떤 클래스에서 올랐는지 표 2를 볼 것 — HYP는 사지유도 전압 기준도 쓰므로 이쪽일 가능성 |
| 비유의 | 검출 실패 | QUICK 표본(클래스당 1,200) 탓일 수 있다. `QUICK=False`로 재실행 |
| < 0 | 예측 반전 | 라벨·정규화 재점검. CELL 4의 레코드 전체 스케일 정규화가 의심 1순위 |

## 이 실험이 남기는 것

1. **실험3이 버린 7,300건의 회수 곡선** — "단일 유도로 못 보는 몫"이 유도를 늘릴 때
   얼마나 돌아오는지의 정량치
2. **lead-agnostic 가중치 1개** (`run.save_model`) — 실험 11(전극위치 augmentation)과
   실험 12(도메인 축 분리)가 그대로 이어받는다
3. **abstain 정책의 근거** — 어떤 유도 구성에서 어떤 라벨군을 포기해야 하는지가
   추정이 아니라 측정으로 정해진다. 실험4의 기권 헤드와 여기서 만난다

## 주의

- **`{II,V1}`의 V1은 실제 텔레메트리의 V1과 다르다.** PTB-XL은 표준 흉부 전극 위치에서
  잰 값이고, 임상 텔레메트리의 modified chest lead(MCL1)는 전극 위치가 다르다.
  → 실험 11(전극위치 augmentation)에서 그 갭을 따로 다룬다. 여기 수치는 **상한**이다.
- **12유도 재구성으로 Tier C를 진단하려 하지 말 것.** 2025년 후속 연구가 축소유도
  재구성을 "평균으로의 회귀라 임상 부적합"이라고 경고했다(`ailab-2026-0016`).
  없는 유도는 **abstain**이 정답이다.
